In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["groq_api_key"]=os.getenv("groq_api_key")

In [3]:
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:openai/gpt-oss-20b")


In [4]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
	title:str=Field(description="The tile of the Movie")
	year:int=Field(descripton="This year the movie was released")
	director:str=Field(description="the character of the movie")
	rating:float=Field(description="The movirs rating out of 10")


model_with_structure=model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about the movie inception")
response

C:\Users\LAVANYA\AppData\Local\Temp\ipykernel_29236\3261787531.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descripton'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year:int=Field(descripton="This year the movie was released")


Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [7]:
from pydantic import BaseModel,Field 
class Movie(BaseModel):
	title:str=Field(...,description="The tile of the Movie")
	year:int=Field(...,descripton="This year the movie was released")
	director:str=Field(...,description="the character of the movie")
	rating:float=Field(...,description="The movirs rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)
response=model_with_structure.invoke("Provide details abiut the movie inception")
response

C:\Users\LAVANYA\AppData\Local\Temp\ipykernel_29236\2771948864.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descripton'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year:int=Field(...,descripton="This year the movie was released")


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user: "Provide details about the movie inception". They likely want details: title, director, rating, year, character? The tool function requires: director, rating, title, year. "character of the movie" maybe "character" field? The function signature: Movie takes director, rating, title, year. So we can call the function with those arguments. We should provide details about the movie. So we need to call function. We\'ll call the function with the movie details. The rating out of 10: Inception rating: 8.8 (on IMDb). Year 2010. Director: Christopher Nolan. So we call function.\n\nWe must respond with the function call.', 'tool_calls': [{'id': 'fc_02256bda-999d-4598-b594-15963e55e1f6', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 152, 't

In [8]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
	name:str
	role:str
class MovieDetails(BaseModel):
	title:str
	year:int
	cast:list[Actor]
	genres:list[str]
	budget:float|None=Field(None,description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide details abiut the movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160000000.0)

In [17]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, "The title of the Movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]
model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke(
    """Provide all details of the movie Avengers.
    Include:
    - title
    - year
    - director
    - rating out of 10
    """
)

print(response)

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2008}


In [21]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b")

In [23]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
	"""Contact information for a person"""
	name:str=Field(description="THe name of the person")
	email:str=Field(description="The email address of the person")
	phone:str=Field(description="The phone number of the person")

agent=create_agent(model, response_format=ContactInfo)

result=agent.invoke({"messages":[{"role":"user","content":"Extract contact info from:johndoe ,john@examole.com, 555 123-4567"}]})
result

{'messages': [HumanMessage(content='Extract contact info from:johndoe ,john@examole.com, 555 123-4567', additional_kwargs={}, response_metadata={}, id='9d8b277a-44b8-41a6-9a3f-0ffc5192e390'),
  AIMessage(content='{"name":"johndoe","email":"john@examole.com","phone":"555 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON object conforming to ContactInfo schema. The input: "johndoe ,john@examole.com, 555 123-4567". So name: "johndoe". Email: "john@examole.com". Phone: "555 123-4567". We must output compact JSON, no extra fields. Ensure fields: name, email, phone. All strings. Output only final JSON object. Let\'s produce:\n\n{"name":"johndoe","email":"john@examole.com","phone":"555 123-4567"}'}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 241, 'total_tokens': 396, 'completion_time': 0.184113424, 'completion_tokens_details': {'reasoning_tokens': 121}, 'prompt_time': 0.012627744, 'prompt_tokens_details': None, 'queue_time': 0.3381

In [26]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo(TypedDict):
	"""Contact information for a person"""
	name:str
	email:str
	phone:str
agent=create_agent(model, response_format=ContactInfo)
result=agent.invoke({"messages":[{"role":"user","content":"Extract contact info from:johndoe ,john@examole.com, 555 123-4567"}]})
result


{'messages': [HumanMessage(content='Extract contact info from:johndoe ,john@examole.com, 555 123-4567', additional_kwargs={}, response_metadata={}, id='231e7733-c029-4277-8805-07d3952dc060'),
  AIMessage(content='{"name":"johndoe","email":"john@examole.com","phone":"555 123-4567"}', additional_kwargs={'reasoning_content': 'We need to produce JSON that matches the schema ContactInfo. Must include all required fields: name, email, phone. The input: "johndoe ,john@examole.com, 555 123-4567". Name: johndoe. Email: john@examole.com. Phone: 555 123-4567. We need compact JSON formatting. Output only JSON object. Ensure it\'s valid JSON. So output:\n\n{"name":"johndoe","email":"john@examole.com","phone":"555 123-4567"}\n\nCheck: The schema expects email, name, phone. Yes. All present. Done.'}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 214, 'total_tokens': 382, 'completion_time': 0.186833143, 'completion_tokens_details': {'reasoning_tokens': 134}, 'prompt_tim